# DataHub Error Case 02 — GMS Critical: Invalid URN

**Monitor:** DataHub GMS Critical  
**Component:** datahub-gms

In [1]:
import sys
sys.path.insert(0, '.')
import importlib
import datahub_test_utils
importlib.reload(datahub_test_utils)
from datahub_test_utils import *

r_login = get_token()
assert r_login.status_code == 200
session_cookie = r_login.cookies.get('actor')
headers = {'Cookie': f'actor={session_cookie}', 'Content-Type': 'application/json'}
print('Logged in')

✅ DataHub test utils loaded
   GMS: http://192.168.139.2:8080
   OpenSearch: https://192.168.194.1:9200
✅ DataHub test utils loaded
   GMS: http://192.168.139.2:8080
   OpenSearch: https://192.168.194.1:9200
Logged in


In [2]:
# Cell 2: Invalid URN format
query = '{ dataset(urn: \"invalid-urn-format-xyz\") { urn } }'
r = requests.post(
    f'{DATAHUB_URL}/api/graphql',
    headers=headers,
    json={'query': query}
)
print(f'Status: {r.status_code}')
resp = r.json()
print(json.dumps(resp.get('errors', resp), indent=2)[:500])

Status: 401
{
  "timestamp": 1781069326432,
  "status": 401,
  "error": "Unauthorized",
  "path": "/api/graphql"
}


In [5]:
# Cell 3: Nonexistent entity URN
query2 = '{ dataset(urn: \"urn:li:dataset:(urn:li:dataPlatform:INVALID,nonexistent,PROD)\") { urn } }'
r2 = requests.post(
    f'{DATAHUB_URL}/api/graphql',
    headers=headers,
    json={'query': query2}
)
print(f'Status: {r2.status_code}')

Status: 401


In [6]:
# Cell 4: Find ERROR logs
wait_for_logs(5)
print('Searching for GMS critical errors...')
hits = search_datahub_errors(minutes_ago=2, container='datahub-gms')
print_logs(hits)

⏳ Waiting 10s for logs to reach OpenSearch...
Searching for GMS critical errors...
✅ Found 4 log entries:

  [2026-06-10T05:29:30] WARN  | datahub-gms | c.d.a.a.f.AuthenticationEnforcementFilter
         Rejected unauthenticated request to protected path: /api/graphql from 192.168.194.1

  [2026-06-10T05:29:21] WARN  | datahub-gms | c.d.a.a.f.AuthenticationEnforcementFilter
         Rejected unauthenticated request to protected path: /api/graphql from 192.168.194.1

  [2026-06-10T05:28:53] WARN  | datahub-gms | c.d.a.a.f.AuthenticationEnforcementFilter
         Rejected unauthenticated request to protected path: /api/graphql from 192.168.194.1

  [2026-06-10T05:28:46] WARN  | datahub-gms | c.d.a.a.f.AuthenticationEnforcementFilter
         Rejected unauthenticated request to protected path: /api/graphql from 192.168.194.1



In [7]:
print('\n=== Summary ===')
print('Error:   GraphQL error / 500')
print('Cause:   Invalid URN, malformed query, missing entity')
print('Monitor: DataHub GMS Critical fires on ERROR count > 0')
print('Fix:     Validate URN format before API calls')
print('Format:  urn:li:dataset:(urn:li:dataPlatform:<p>,<name>,<env>)')


=== Summary ===
Error:   GraphQL error / 500
Cause:   Invalid URN, malformed query, missing entity
Monitor: DataHub GMS Critical fires on ERROR count > 0
Fix:     Validate URN format before API calls
Format:  urn:li:dataset:(urn:li:dataPlatform:<p>,<name>,<env>)
